# CS-Jail-UR: Phase 2 (Exp 7 training + Exp 8 generation, GPU)

Runs headless with **Save Version → Save & Run All (Commit)**. Design: `docs/PHASE2_PLAN.md`;
driver: `scripts/phase2_kaggle.py`.

| `STAGE` | What it does | Needs | Time |
|---|---|---|---|
| `"smoke"` | DPO on 8 harmless pairs per model (fp16 on T4, LoRA modules), then the adapter in vLLM | nothing private | ~30–40 min |
| `"train_eval"` | All adapters (C and B_ext × 3 seeds, C n-curve, D6 ablation) and Exp 8 generations, Phi-3 on GPU 0 and Llama on GPU 1 in parallel; resumable | the private pair dataset | ~4–6 h |

Before saving:
1. Settings → Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons → Secrets: attach **HF_TOKEN** (Llama-3.2 is gated).
3. For `"train_eval"`: Add Input → your **private** dataset made from `outputs/phase2_upload.zip`
   (verified pairs + external English pairs; harmful text). Any dataset name works: the notebook
   searches `/kaggle/input` for it.

Nothing is judged here (Kaggle cannot reach the signed-in Ollama judge). Download
`phase2_outputs.zip` (and `phase2_adapters.zip` if wanted) from the version's **Output** tab and judge
on the laptop with `exp8_posteval.py --judge-only`. Keep this notebook **private**.


In [ ]:
# Set before "Save & Run All".
STAGE = "smoke"                          # "smoke" first; "train_eval" after the pair review is applied
MODELS = ["phi3", "llama32"]             # the Phase 2 models (one GPU each in train_eval)
KEEP_ADAPTERS = True                     # also zip the trained LoRA adapters (~40 MB each)

import glob, json, os, subprocess, time
from IPython import get_ipython


def sh(cmd):
    """Run a shell command with streamed output; stop the whole run if it fails."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")


def package():
    """Zip results so far into /kaggle/working (overwrites): generations, manifests, smoke results."""
    keep = [p for p in ("outputs/phase2_smoke", "outputs/phase2_gate", "outputs/exp8") if os.path.exists(p)]
    keep += glob.glob("outputs/models/*/training_manifest.json")
    if keep:
        subprocess.run("rm -f /kaggle/working/phase2_outputs.zip && zip -qr /kaggle/working/phase2_outputs.zip "
                       + " ".join(keep) + " -x 'outputs/phase2_smoke/*/adapter/*.safetensors'", shell=True)
    if KEEP_ADAPTERS and os.path.isdir("outputs/models"):
        subprocess.run("rm -f /kaggle/working/phase2_adapters.zip && zip -qr /kaggle/working/phase2_adapters.zip outputs/models",
                       shell=True)


In [ ]:
# Repo + Python 3.12 environment for the pinned stack (Kaggle's own Python is 3.13).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
# The training stack (TRL, PEFT, bitsandbytes) on top of the pinned vLLM stack.
sh('python -m uv pip install --python /root/py312/bin/python -e "' + '.[train,dev]' + '"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# GPU, gated-model token, frozen dataset, test suite. Any failure stops the run here.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("attach the HF_TOKEN secret: Llama-3.2 is gated")
sh("nvidia-smi -L")
sh("python scripts/exp0_finalize_data.py --restore")
sh("python -m pytest tests -q -p no:cacheprovider")


In [ ]:
# GPU smoke (PROTOCOL §9): fp16 DPO on 8 harmless pairs, then the adapter in vLLM, per model.
if STAGE == "smoke":
    for m in MODELS:
        sh(f"python scripts/phase2_kaggle.py smoke --model {m} --step train")
        sh(f"python scripts/phase2_kaggle.py smoke --model {m} --step serve")
        package()
    print("SMOKE PASSED for", MODELS)


In [ ]:
# Exp 7 + Exp 8 generation: one model per GPU, in parallel, resumable; packaged every 10 minutes.
if STAGE == "train_eval":
    # Finds the private bundle anywhere under /kaggle/input (zip or unpacked, any folder name/depth).
    sh("python scripts/phase2_kaggle.py stage")
    sh("ls outputs/exp6 && cat outputs/exp6/review/verified_summary.json")

    def run_parallel(phase, epochs):
        """One model per GPU; package every 10 minutes; stop the run if a model fails."""
        procs = {}
        for gpu, m in enumerate(MODELS):
            env_m = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
            procs[m] = subprocess.Popen(["python", "scripts/phase2_kaggle.py", "run", "--model", m, "--phase", phase,
                                         "--epochs", str(epochs)], env=env_m,
                                        stdout=open(f"/kaggle/working/phase2_{m}_{phase}_e{epochs}.log", "w"),
                                        stderr=subprocess.STDOUT)
        while any(p.poll() is None for p in procs.values()):
            time.sleep(60 if phase == "gate" else 600)
            package()
            for m in MODELS:
                print(f"--- {m} {phase} e{epochs} ---", flush=True)
                subprocess.run(f"tail -n 2 /kaggle/working/phase2_{m}_{phase}_e{epochs}.log", shell=True)
        package()
        bad = {m: p.returncode for m, p in procs.items() if p.returncode}
        if bad:
            for m in bad:   # the real error is in the log, not in this cell's traceback
                print(f"=== {m} {phase} e{epochs}: last 80 log lines ===", flush=True)
                subprocess.run(f"tail -n 80 /kaggle/working/phase2_{m}_{phase}_e{epochs}.log", shell=True)
            raise RuntimeError(f"{phase} failed: {bad}; see /kaggle/working/phase2_<model>_{phase}_e{epochs}.log")

    # Learning check (configs/dpo.yaml learning_check, declared before training): seed-42 C and B_ext
    # adapters on their TRAINING logs; climb the epochs ladder until all pass (or its top).
    ladder = json.loads(subprocess.run(["python", "-c", "import yaml, json; print(json.dumps(yaml.safe_load("
                                        "open('configs/dpo.yaml'))['learning_check']['epochs_ladder']))"],
                                       capture_output=True, text=True, check=True).stdout)
    for epochs in ladder:
        run_parallel("gate", epochs)
        decided = subprocess.run(["python", "scripts/phase2_kaggle.py", "decide", "--models", *MODELS,
                                  "--epochs", str(epochs)]).returncode
        if decided == 0:
            break
        print(f"LEARNING CHECK FAILED at {epochs} epochs" + ("; retraining at the next value" if epochs != ladder[-1]
              else "; top of the ladder reached, continuing with it (reported)"), flush=True)
    print(f"USING {epochs} EPOCHS FOR ALL PHASE 2 ADAPTERS", flush=True)
    run_parallel("rest", epochs)
    print("TRAIN_EVAL DONE")


In [ ]:
package()
sh("ls -lh /kaggle/working/*.zip")
print("DONE: download phase2_outputs.zip (and phase2_adapters.zip) from this version's Output tab.")
